# Airline Operations & Disruption Intelligence — Business Insights

## 1. Import libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Load the final enriched dataset

This file was created after:

**BTS cleaning → airport reference → Open-Meteo weather integration**


In [2]:
file_path = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\flights_2026_q1.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Data loaded successfully")
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


ParserError: Error tokenizing data. C error: out of memory

In [ ]:
df.head()

## 3. Overall operational performance

### Business question
**What is the overall operational condition of the flights?**


In [ ]:
total_flights = len(df)
cancelled_flights = df["cancelled"].sum()
diverted_flights = df["diverted"].sum()

completed_df = df[
    (df["cancelled"] == 0) &
    (df["diverted"] == 0)
]

delay_rate = completed_df["arr_del15"].mean() * 100
cancellation_rate = cancelled_flights / total_flights * 100
diversion_rate = diverted_flights / total_flights * 100

print("Total flights:", f"{total_flights:,}")
print("Completed flights:", f"{len(completed_df):,}")
print("Arrival delay rate:", round(delay_rate, 2), "%")
print("Cancellation rate:", round(cancellation_rate, 2), "%")
print("Diversion rate:", round(diversion_rate, 2), "%")


### Key Finding

> During the study period, the dataset contained **[18,47,242] flights**. The arrival delay rate was **[21.59]%**, cancellation rate was **[3.37]%**, and diversion rate was **[0.26]%**.

This gives management the starting point for understanding operational performance.


## 4. Delay performance

### Business question
**How large are the delays?**


In [ ]:
avg_departure_delay = completed_df["dep_delay"].mean()
median_departure_delay = completed_df["dep_delay"].median()
avg_arrival_delay = completed_df["arr_delay"].mean()
median_arrival_delay = completed_df["arr_delay"].median()

print("Average departure delay:", round(avg_departure_delay, 2), "minutes")
print("Median departure delay:", round(median_departure_delay, 2), "minutes")
print("Average arrival delay:", round(avg_arrival_delay, 2), "minutes")
print("Median arrival delay:", round(median_arrival_delay, 2), "minutes")


### Key Finding

If the average is much higher than the median, some severe delays are increasing the average.

**Business impact:** severe delays can create downstream disruption for later flights and passengers.


## 5. Monthly operational performance

### Business question
**Is operational performance changing over time?**


In [ ]:
df["fl_date"] = pd.to_datetime(df["fl_date"])

monthly_summary = (
    df.groupby(df["fl_date"].dt.to_period("M"))
    .agg(
        flights=("fl_date", "size"),
        cancelled=("cancelled", "sum"),
        diverted=("diverted", "sum")
    )
)

monthly_summary["cancellation_rate"] = monthly_summary["cancelled"] / monthly_summary["flights"] * 100
monthly_summary["diversion_rate"] = monthly_summary["diverted"] / monthly_summary["flights"] * 100

monthly_summary


## 6. Airline performance

### Business question
**Which airlines show higher operational risk?**

Use a minimum flight count so small airlines do not dominate the ranking.


In [ ]:
airline_summary = (
    df.groupby("mkt_carrier")
    .agg(
        flights=("mkt_carrier", "size"),
        avg_arrival_delay=("arr_delay", "mean"),
        delayed_flights=("arr_del15", "sum"),
        cancelled=("cancelled", "sum"),
        diverted=("diverted", "sum")
    )
)

airline_summary["delay_rate"] = df.groupby("mkt_carrier")["arr_del15"].mean() * 100
airline_summary["cancellation_rate"] = airline_summary["cancelled"] / airline_summary["flights"] * 100
airline_summary["diversion_rate"] = airline_summary["diverted"] / airline_summary["flights"] * 100

airline_summary = airline_summary[airline_summary["flights"] >= 100]
airline_summary.sort_values("delay_rate", ascending=False).head(10)


### Business interpretation

Airlines at the top have higher observed delay rates among airlines with sufficient flight volume.

**Recommendation:** investigate airport mix, route mix, schedule timing, and recorded delay causes before deciding what is driving the difference.


## 7. Airport performance

### Business question
**Where are operational problems concentrated?**


In [ ]:
airport_summary = (
    df.groupby("origin")
    .agg(
        flights=("origin", "size"),
        avg_arrival_delay=("arr_delay", "mean"),
        delayed_flights=("arr_del15", "sum"),
        cancelled=("cancelled", "sum"),
        diverted=("diverted", "sum")
    )
)

airport_summary["delay_rate"] = df.groupby("origin")["arr_del15"].mean() * 100
airport_summary["cancellation_rate"] = airport_summary["cancelled"] / airport_summary["flights"] * 100

airport_summary = airport_summary[airport_summary["flights"] >= 1000]
airport_summary.sort_values("avg_arrival_delay", ascending=False).head(10)


### Key Finding

High-delay airports are operational hotspots.

**Recommendation:** investigate these airports using departure hour, airline, route, delay cause, and weather information.


## 8. Route performance

### Business question
**Are delays concentrated on specific routes?**


In [ ]:
df["route"] = df["origin"] + " → " + df["dest"]

route_summary = (
    df.groupby("route")
    .agg(
        flights=("route", "size"),
        avg_arrival_delay=("arr_delay", "mean"),
        delay_rate=("arr_del15", "mean")
    )
)

route_summary["delay_rate"] *= 100
route_summary = route_summary[route_summary["flights"] >= 500]

route_summary.sort_values("delay_rate", ascending=False).head(10)


### Key Finding

High-delay routes should be investigated using airport, airline, schedule, and delay-cause information.

The minimum-flight rule prevents a route with very few flights from producing a misleading ranking.


## 9. Delay causes

### Business question
**What contributes the most recorded delay minutes?**


In [ ]:
delay_columns = {
    "Carrier Delay": "carrier_delay",
    "Weather Delay": "weather_delay",
    "NAS Delay": "nas_delay",
    "Security Delay": "security_delay",
    "Late Aircraft Delay": "late_aircraft_delay"
}

delay_totals = {}

for cause, column in delay_columns.items():
    if column in df.columns:
        delay_totals[cause] = df[column].fillna(0).sum()

delay_totals = pd.Series(delay_totals).sort_values(ascending=False)

print(delay_totals)
print("Largest recorded delay cause:", delay_totals.idxmax())


### Key Finding

> **[Largest cause]** contributed the most recorded delay minutes.

**Recommendation:** prioritize investigation of the largest recorded delay contributors.

Important: BTS recorded delay causes and observed weather conditions are different concepts.


## 10. Weather impact

### Business question
**Are disruptions different under bad-weather conditions?**

The statistics notebook already tested weather relationships. Here we translate those results into business language.


In [ ]:
if "bad_weather" in df.columns:
    weather_summary = (
        df.groupby("bad_weather")
        .agg(
            flights=("bad_weather", "size"),
            delay_rate=("arr_del15", "mean"),
            cancellation_rate=("cancelled", "mean"),
            diversion_rate=("diverted", "mean")
        )
    )

    weather_summary["delay_rate"] *= 100
    weather_summary["cancellation_rate"] *= 100
    weather_summary["diversion_rate"] *= 100

    weather_summary.index = [
        "Normal Weather" if x == 0 else "Bad Weather"
        for x in weather_summary.index
    ]

    weather_summary
else:
    print("Column 'bad_weather' was not found.")


### Key Finding

If bad-weather rates are higher:

> Flights operating under bad-weather conditions showed higher observed disruption levels than flights under normal weather.

**Business action:** monitor weather-sensitive airports more closely and improve disruption planning during poor-weather periods.

### Important
Use **"associated with"**, not **"caused by"**.

The analysis does not by itself prove causation.


## 11. Statistics — Key Finding

The completed statistics analysis tested:

1. Mean arrival delay vs 0
2. Weekday vs weekend departure delay
3. Cancellation vs airline
4. Arrival delay across airlines
5. Departure delay vs arrival delay
6. Cancellation vs bad weather
7. Diversion vs bad weather

The project uses **α = 0.05** and emphasizes that statistical significance is not the same as business significance. fileciteturn10file0L1-L8

Important completed results:

- Mean arrival delay ≈ **7.25 minutes**
- Weekday mean departure delay ≈ **13.12 minutes**
- Weekend mean departure delay ≈ **15.62 minutes**
- Pearson correlation between departure and arrival delay ≈ **0.9693**
- Bad weather and cancellation showed statistically significant association.
- Bad weather and diversion showed statistically significant association.

## 12. Strong business insights

### Insight 1 — Overall Operational Performance

**Finding:**  
During January–March 2026, the dataset contains **1,847,242 flights**. Out of these, **1,780,194 flights were completed**. The overall arrival delay rate was **21.59%**, while **3.37% of flights were cancelled** and **0.26% were diverted**.

**Evidence:**
- **Total flights:** 1,847,242
- **Completed flights:** 1,780,194
- **Arrival delay rate:** 21.59%
- **Cancellation rate:** 3.37%
- **Diversion rate:** 0.26%

**Business impact:**  
The results show that **flight delays were a larger operational issue than cancellations or diversions** during the study period. Therefore, improving delay performance should be a major operational focus.

**Recommendation:**  
Operations teams should investigate where and when delays are concentrated, particularly by:

- Airport
- Airline
- Route
- Departure hour
- Day of week
- Recorded delay cause
- Weather condition

This can help identify the specific operational areas contributing most to disruption.



---

### Insight 2 — Departure delays carry into arrival
**Finding:** Departure and arrival delay have a very strong positive linear relationship.

**Evidence:** Pearson correlation ≈ **0.9693**.

**Business impact:** delays at departure can continue into arrival performance.

**Recommendation:** focus on preventing and reducing departure delays.

---

### Insight 3 — Weekend operations
**Finding:** Weekend average departure delay was about **15.62 minutes**, compared with **13.12 minutes** on weekdays.

**Business impact:** weekend operations may require additional attention.

**Recommendation:** review weekend scheduling, airport workload, and delay causes.

---

### Insight 4 — Airline differences
**Finding:** Airlines show different observed delay and cancellation patterns.

**Recommendation:** investigate the highest-risk airlines after considering flight volume, airport mix, routes, and delay causes.

---

### Insight 5 — Airport hotspots
**Finding:** Some airports show higher observed delay levels.

**Recommendation:** prioritize consistently underperforming airports for deeper operational review.

---

### Insight 6 — Delay drivers
**Finding:** [Largest recorded delay cause] contributed the most recorded delay minutes.

**Recommendation:** prioritize root-cause investigation in this area.

---

### Insight 7 — Weather and disruption
**Finding:** Bad-weather status was statistically associated with cancellation and diversion.

**Recommendation:** increase monitoring and disruption planning at weather-sensitive airports.

**Important:** association is not proof that weather alone caused the disruption.
